# RESource Sensitivity Analysis
**Manuscript:** *Mapping feasible renewable transition space: land-use, conservation,
and grid-access constraints on wind and solar in British Columbia*

Provides:
- Grid-connection cost sensitivity
- Capacity density and economic-parameter sensitivity

This notebook loads your exported cluster CSVs and re-derives the screening-level
LCOE proxy under parameter variants, exactly replicating the formula in
`RES/score.py::CellScorer.calculate_score()`.


## 0 · Configuration
Edit the paths and parameters below before running.

In [ ]:
weather_year='2024'
run_date='20260603'
scenario_name='BASELINE'

- Set Root

In [ ]:
from pathlib import Path
current_dir=Path.cwd()
# Go up 2 folders
root = current_dir.parent.parent
print('Root:', root)

paper_resources=current_dir
print('Paper Resources:', paper_resources)
country_kwd='Canada'
region_code='BC'


In [ ]:
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

import RESource.visuals as vis
from RESource import utility as utils
from RESource.hdf5_handler import DataHandler
from RESource.CellCapacityProcessor import get_sub_nationally_aggregated_capacity
plt.style.use(root / 'RES' / 'visual_styles' / 'elsevier.mplstyle')
# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
from pathlib import Path
from RESource import utility as utils

# ── Cluster CSV paths ─────────────────────────────────────────────────────
# Adjust <RUN_ID> to match your actual run folder, e.g. BASELINE_20250520
RUN_ID = f'{scenario_name.upper()}_{weather_year}_{run_date}'

vis_save_to_root=utils.ensure_path(paper_resources / f"vis/{country_kwd}/{region_code}/{RUN_ID}")
results_save_to_root=utils.ensure_path(paper_resources / f"results/{country_kwd}/{region_code}/{RUN_ID}")
cluster_source=root/f"results/Canada/BC/{scenario_name}_{weather_year}_{run_date}/clusters"

SOLAR_CSV = cluster_source / f"resource_options_solar_British Columbia_{weather_year}.csv"
WIND_CSV  = cluster_source / f"resource_options_wind_British Columbia_{weather_year}.csv"  # Assuming wind clusters are saved with the same name; adjust if different


In [ ]:

# ── Transmission cost parameters (from your BC config) ────────────────────
# capacity_disaggregation.transmission.grid_connection_cost_per_Km
GCC_BASELINE   = 2.6    # M$/km  — MISO spur-line proxy
TX_REBUILD     = 0.56   # M$     — fixed substation upgrade cost

# ── Financial baseline ────────────────────────────────────────────────────
BASELINE_RATE  = 0.07   # discount rate used in the published run
LCOE_THRESHOLD = 90.0  # $/MWh — cut-off for 'economically accessible' capacity

# ── Output directories ────────────────────────────────────────────────────
sensitivity_data_save_to = Path(f"results/sensitivity/{RUN_ID}")
sensitivity_data_save_to.mkdir(parents=True, exist_ok=True)

print("Solar CSV exists:", SOLAR_CSV.exists())
print("Wind  CSV exists:", WIND_CSV.exists())


## 1 · Imports

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import spearmanr

# Add the codebase root to path so we can import the sensitivity modules
sys.path.insert(0, str(Path(".").resolve()))

from notebooks.Publication_resources.sensitivity_grid_cost import (
    get_crf, compute_lcoe,
    run_grid_cost_sensitivity,
    plot_supply_curves, plot_spearman,
)
from notebooks.Publication_resources.sensitivity_economic_params import (
    run_oat_sensitivity, run_interaction_sweep,
    plot_tornado, plot_interaction_heatmap,
    DENSITY_SCENARIOS, DISCOUNT_RATE_SCENARIOS,
    CAPEX_MULTIPLIER_SCENARIOS,
)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 130, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False})
print("Imports OK")


## 2 · Load cluster data

The cluster CSV exported by `RESources_builder.export_results()` contains one row
per cluster with columns: `cluster_id, lcoe, capex, fom, vom, CF_mean,
potential_capacity, nearest_station_distance_km, Operational_life, resource_type`.

> **Note:** The `lcoe` column is the *normalised* LCOE (1 MW reference capacity)
> used for clustering. The sensitivity scripts recompute *actual-capacity* LCOE
> (matching `lcoe_actualCap_{resource_type}` in `score.py`) which accounts for
> economies-of-scale in grid connection costs.


In [ ]:
def load_cluster_csv(path: Path, resource_type: str) -> pd.DataFrame:
    """Load and validate a cluster CSV."""
    required = ["cluster_id", "potential_capacity", "CF_mean", "capex",
                "fom", "vom", "nearest_station_distance_km", "Operational_life"]
    df = pd.read_csv(path)
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {path.name}: {missing}")
    print(f"{resource_type:5s} | {len(df):4d} clusters | "
          f"total {df['potential_capacity'].sum()/1e3:.1f} GW | "
          f"LCOE range {df['lcoe'].min():.1f}–{df['lcoe'].max():.1f} $/MWh")
    return df

dfs = {}
for rtype, csv_path in [("solar", SOLAR_CSV), ("wind", WIND_CSV)]:
    if csv_path.exists():
        dfs[rtype] = load_cluster_csv(csv_path, rtype)
    else:
        print(f"SKIP {rtype}: {csv_path} not found")

if dfs:
    next(iter(dfs.values())).head(5)


### 2a · Sanity-check: reproduce LCOE for the top-ranked cluster

Pick the highest-ranked (lowest `lcoe`) cluster and verify the formula gives
a sensible result. The re-derived value will differ from the CSV's `lcoe` column
because that column is the *normalised* LCOE; this cell computes the
*actual-capacity* LCOE used for supply-curve ordering.


In [ ]:
for rtype, df in dfs.items():
    row  = df.sort_values("lcoe").iloc[0]
    N    = int(row["Operational_life"])
    crf  = get_crf(BASELINE_RATE, N)
    lcoe = compute_lcoe(
        capacity_mw       = row["potential_capacity"],
        cf_mean           = row["CF_mean"],
        capex_musd_per_mw = row["capex"],
        fom_musd_per_mw   = row["fom"],
        vom_musd_per_mwh  = row["vom"],
        distance_km       = row["nearest_station_distance_km"],
        gcc_musd_per_km   = GCC_BASELINE,
        tx_rebuild_musd   = TX_REBUILD,
        crf               = crf,
    )
    print(f"{rtype:5s} best cluster: {row['cluster_id']}")
    print(f"       normalised LCOE (from CSV):      {row['lcoe']:.2f} $/MWh")
    print(f"       actual-capacity LCOE (recomputed): {lcoe:.2f} $/MWh")
    print(f"       CF={row['CF_mean']:.3f}, cap={row['potential_capacity']:.0f} MW, "
          f"dist={row['nearest_station_distance_km']:.1f} km")
    print()


---
## 3 · M2 — Grid-connection cost sensitivity

### Rationale
The spur-line cost estimate (2.6 M$/km) derives from MISO (US Midwest) data and
may not reflect Canadian terrain or right-of-way conditions (R1.3, R2.M1).
To quantify this uncertainty, we apply a terrain-routing detour multiplier
κ to the effective spur-line distance:

$$\text{gcc\_effective} = \kappa \times \text{gcc\_baseline}$$

κ < 1 represents cheaper/shorter routes; κ > 1 represents terrain detours or
cost over-runs.  We test κ ∈ {0.6, 0.8, 1.0, 1.3, 1.6, 2.0}.

### Key metrics
1. **Spearman ρ** of cluster ranking vs. baseline — if ρ > 0.90, rank order is robust
2. **Supply-curve bands** — envelope of cumulative developable potential under best/worst assumptions


In [ ]:
KAPPA_VALUES = [0.6, 0.8, 1.0, 1.3, 1.6, 2.0]
grid_sensitivity_results = {}

for rtype, df in dfs.items():
    print(f"Running M2 for {rtype}...")
    grid_sensitivity_results[rtype] = run_grid_cost_sensitivity(
        df             = df,
        resource_type  = rtype,
        baseline_gcc   = GCC_BASELINE,
        baseline_tx    = TX_REBUILD,
        interest_rate  = BASELINE_RATE,
        kappa_values   = KAPPA_VALUES,
        lcoe_threshold = LCOE_THRESHOLD,
    )
    # Save CSVs
    grid_sensitivity_results[rtype]["spearman"].to_csv(sensitivity_data_save_to / f"M2_spearman_{rtype}.csv", index=False)
    grid_sensitivity_results[rtype]["summary"].to_csv(sensitivity_data_save_to / f"M2_summary_{rtype}.csv",  index=False)
    print(f"  done — saved to {sensitivity_data_save_to}")


### 3a · Spearman rank-correlation table

In [ ]:
for rtype, res in grid_sensitivity_results.items():
    print(f"\n{'─'*50}")
    print(f"  {rtype.upper()} — Spearman ρ (rank correlation vs. κ=1.0 baseline)")
    print(f"{'─'*50}")
    display(res["spearman"][["kappa", "gcc_musd_per_km", "spearman_rho", "p_value"]]
            .style.format({"kappa": "{:.1f}", "gcc_musd_per_km": "{:.2f}",
                           "spearman_rho": "{:.4f}", "p_value": "{:.4f}"})
            .bar(subset=["spearman_rho"], color="#90CAF9", vmin=0, vmax=1))


### 3b · Supply-curve bands

In [ ]:
fig, axes = plt.subplots(1, len(dfs), figsize=(10 , 4), sharey=False, dpi=500)
if len(dfs) == 1:
    axes = [axes]

cmap = plt.cm.RdYlGn_r
colours = cmap(np.linspace(0.1, 0.9, len(KAPPA_VALUES)))

for ax, (rtype, res) in zip(axes, grid_sensitivity_results.items()):
    for kappa, colour in zip(KAPPA_VALUES, colours):
        cum_gw, lcoe_arr = res["curves"][kappa]
        lw = 2.5 if kappa == 1.0 else 1.0
        ls = "-"  if kappa == 1.0 else "--"
        lbl = f"κ={kappa:.1f} (baseline)" if kappa == 1.0 else f"κ={kappa:.1f}"
        ax.step(cum_gw, lcoe_arr, where="post", color=colour,
                linewidth=lw, linestyle=ls, label=lbl)
    ax.axhline(LCOE_THRESHOLD, color="grey", linestyle=":", linewidth=1,
               label=f"{LCOE_THRESHOLD:.0f} $/MWh threshold")
    ax.set_xlabel("Cumulative developable potential (GW)", fontsize=11)
    ax.set_ylabel("Screening-level LCOE proxy ($/MWh)", fontsize=11)
    ax.set_title(f"{rtype.title()} — Grid-cost sensitivity", fontsize=12)
    ax.legend(fontsize=10.5, loc="upper left")
    ax.set_ylim(0, LCOE_THRESHOLD * 2.5)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f"))

fig.tight_layout()
fig.savefig(sensitivity_data_save_to / "M2_supply_curves_combined.svg", dpi=500)
plt.show()
print(f"Figure saved to {sensitivity_data_save_to / 'M2_supply_curves_combined.svg'}")


### 3c · Accessible capacity at LCOE threshold

In [ ]:
rows = []
for rtype, res in grid_sensitivity_results.items():
    for kappa, gw in res["threshold"].items():
        rows.append({"resource_type": rtype, "kappa": kappa,
                     "accessible_gw": round(gw, 2)})

threshold_df = pd.DataFrame(rows).pivot(
    index="kappa", columns="resource_type", values="accessible_gw")
print(f"Accessible capacity (GW) with LCOE ≤ {LCOE_THRESHOLD} $/MWh:")
display(threshold_df.style.format("{:.1f}")
        .background_gradient(cmap="RdYlGn", axis=None))
threshold_df.to_csv(sensitivity_data_save_to / "M2_threshold_capacity.csv")


---
## 4 · M3 — Capacity density and economic-parameter sensitivity

### Parameters swept (one-at-a-time, OAT)

| Parameter | Low | Mid (baseline) | High |
|-----------|-----|----------------|------|
| Solar density (MW/km²) | 1.10 | **1.45** | 1.80 |
| Wind density (MW/km²)  | 2.25 | **3.00** | 3.75 |
| Discount rate (%)      | 5    | **7**    | 10   |
| CAPEX (ATB scenario)   | 0.8× | **1.0×** | 1.2× |
| Fixed O&M              | 0.8× | **1.0×** | 1.2× |

Each parameter is varied while all others remain at baseline.
The tornado chart shows which parameter drives the most uncertainty in total
accessible capacity (GW) at the chosen LCOE threshold.


In [ ]:
m3_oat         = {}
m3_interaction = {}

for rtype, df in dfs.items():
    print(f"Running M3 OAT for {rtype}...")
    m3_oat[rtype] = run_oat_sensitivity(
        df             = df,
        resource_type  = rtype,
        gcc_baseline   = GCC_BASELINE,
        tx_baseline    = TX_REBUILD,
        lcoe_threshold = LCOE_THRESHOLD,
        baseline_rate  = BASELINE_RATE,
    )
    m3_oat[rtype].to_csv(sensitivity_data_save_to / f"M3_oat_{rtype}.csv", index=False)

    print(f"Running M3 interaction sweep for {rtype}...")
    m3_interaction[rtype] = run_interaction_sweep(
        df             = df,
        resource_type  = rtype,
        gcc_baseline   = GCC_BASELINE,
        tx_baseline    = TX_REBUILD,
        lcoe_threshold = LCOE_THRESHOLD,
    )
    m3_interaction[rtype].to_csv(sensitivity_data_save_to / f"M3_interaction_{rtype}.csv", index=False)
    print(f"  done — saved to {sensitivity_data_save_to / f'M3_interaction_{rtype}.csv'}")


### 4a · OAT results table

In [ ]:
for rtype, oat in m3_oat.items():
    baseline_gw = oat["baseline_gw"].iloc[0]
    print(f"\n{'─'*60}")
    print(f"  {rtype.upper()} — baseline accessible capacity: {baseline_gw:.1f} GW "
          f"(LCOE ≤ {LCOE_THRESHOLD} $/MWh)")
    print(f"{'─'*60}")
    display(oat[["parameter", "scenario_label", "scenario_value",
                 "total_gw", "delta_gw", "delta_pct"]]
            .style.format({"scenario_value": "{:.2f}", "total_gw": "{:.1f}",
                           "delta_gw": "{:+.1f}", "delta_pct": "{:+.1f}%"})
            .applymap(lambda v: "color: #c62828" if isinstance(v, str) and v.startswith("-")
                      else ("color: #2e7d32" if isinstance(v, str) and v.startswith("+") else ""),
                      subset=["delta_pct"]))


### 4b · Tornado charts

In [ ]:
# ── Global parameter order — sorted by combined range across all resources ────
all_bars = {}
for rtype, oat in m3_oat.items():
    for param in oat["parameter"].unique():
        sub = oat[oat["parameter"] == param]
        lo  = sub[sub["scenario_label"] == "Low"]["delta_gw"].values
        hi  = sub[sub["scenario_label"] == "High"]["delta_gw"].values
        if len(lo) and len(hi):
            all_bars.setdefault(param, 0)
            all_bars[param] += abs(hi[0]) + abs(lo[0])   # accumulate range

# Ascending → least influential at top, most at bottom (tornado convention)
ordered_params = sorted(all_bars, key=all_bars.get)[::-1]
# ordered_params is now the shared y-axis order for every subplot

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(m3_oat), figsize=(14, 3), dpi=300)
if len(m3_oat) == 1:
    axes = [axes]

for ax, (rtype, oat) in zip(axes, m3_oat.items()):

    # Build lookup for this resource
    bar_lookup = {}
    for param in oat["parameter"].unique():
        sub = oat[oat["parameter"] == param]
        lo  = sub[sub["scenario_label"] == "Low"]["delta_gw"].values
        hi  = sub[sub["scenario_label"] == "High"]["delta_gw"].values
        if len(lo) and len(hi):
            bar_lookup[param] = {"low": lo[0], "high": hi[0]}

    for i, param in enumerate(ordered_params):
        row = bar_lookup.get(param, {"low": 0, "high": 0})

        if row["high"] > 0:
            ax.barh(i, row["high"], left=0, color="#1976D2", alpha=0.85, height=0.5)
            ax.text(row["high"] + 0.05, i, f"+{row['high']:.1f}",
                    va="center", fontsize=9)
        if row["low"] < 0:
            ax.barh(i, row["low"], left=0, color="#D32F2F", alpha=0.85, height=0.5)
            ax.text(row["low"] - 0.05, i, f"{row['low']:.1f}",
                    va="center", ha="right", fontsize=9)

    ax.set_yticks(range(len(ordered_params)))
    ax.set_yticklabels(ordered_params, fontsize=10)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("ΔAccessible capacity vs. baseline (GW)", fontsize=12)
    ax.set_title(
        f"{rtype.title()} — Parameter sensitivity tornado"
        f"(LCOE ≤ {LCOE_THRESHOLD:.0f} $/MWh)", fontsize=12
    )

fig.tight_layout()
fig.savefig(sensitivity_data_save_to / "M3_tornado_combined.jpg", dpi=300)
plt.show()

### 4c · CAPEX × discount rate interaction heatmap

This two-parameter sweep captures the joint effect of cost-scenario
(ATB Low/Mid/High) and financial assumptions, which are correlated in practice
(optimistic cost projections tend to co-occur with favourable financing).


In [ ]:

import matplotlib.patheffects as pe


n_rtypes = len(m3_interaction)
fig, axes = plt.subplots(1, n_rtypes, figsize=(6 * n_rtypes, 3),dpi=500)
if n_rtypes == 1:
    axes = [axes]

row_order = ["Low", "Mid", "High"]
col_order = ["Low", "Mid", "High"]

for ax, (rtype, idf) in zip(axes, m3_interaction.items()):
    pivot = idf.pivot(index="rate_label", columns="capex_label",
                      values="total_gw").reindex(index=row_order, columns=col_order)
    im = ax.imshow(pivot.values, cmap="RdYlGn", aspect="auto",
                   vmin=pivot.values.min() * 0.92,
                   vmax=pivot.values.max() * 1.08)
    plt.colorbar(im, ax=ax, label="Accessible capacity (GW)", shrink=0.85)
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels([f"CAPEX {c}" for c in col_order], fontsize=12)
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels([f"Rate {r}" for r in row_order], fontsize=12)
    ax.set_title(f"{rtype.title()} — CAPEX × discount rate"
                 f"(GW at LCOE ≤ {LCOE_THRESHOLD:.0f} $/MWh)", fontsize=12)
    # for i in range(3):
    #     for j in range(3):
    #         ax.text(j, i, f"{pivot.values[i, j]:.1f}",
    #                 ha="center", va="center", fontsize=12,
    #                 color="white" if pivot.values[i, j] < pivot.values.mean() else "black")
    for i in range(3):
        for j in range(3):
            val = pivot.values[i, j]
            
            # Map value to [0,1] in the colormap range, then get RGB luminance
            norm_val = (val - pivot.values.min() * 0.92) / (
                        pivot.values.max() * 1.08 - pivot.values.min() * 0.92)
            r, g, b, _ = plt.cm.RdYlGn(norm_val)
            luminance  = 0.2126 * r + 0.7152 * g + 0.0722 * b   # WCAG formula
            text_color = "black" if luminance > 0.45 else "white"

            ax.text(j, i, f"{val:.1f}",
                    ha="center", va="center", fontsize=12,
                    fontweight="bold", color=text_color)

fig.tight_layout()
fig.savefig(sensitivity_data_save_to / "M3_interaction_heatmap_combined.svg", dpi=500)
plt.show()
print(f"Figure saved to {sensitivity_data_save_to / 'M3_interaction_heatmap_combined.svg'}")


---
## 5 · Summary statistics for manuscript text

Generates the key numbers you need to report in the revised manuscript and
Response to Reviewers.


In [ ]:
print("=" * 65)
print("  KEY NUMBERS FOR MANUSCRIPT / RESPONSE TO REVIEWERS")
print("=" * 65)

for rtype in dfs:
    print(f"\n── {rtype.upper()} ──────────────────────────────────────────")

    # M2 summary
    if rtype in grid_sensitivity_results:
        sp = grid_sensitivity_results[rtype]["spearman"]
        rho_min = sp["spearman_rho"].min()
        rho_max = sp["spearman_rho"].max()
        kappa_worst = sp.loc[sp["spearman_rho"].idxmin(), "kappa"]
        gw_base = grid_sensitivity_results[rtype]["threshold"][1.0]
        gw_worst = min(grid_sensitivity_results[rtype]["threshold"].values())
        gw_best  = max(grid_sensitivity_results[rtype]["threshold"].values())
        pct_range = 100 * (gw_best - gw_worst) / (gw_base + 1e-9)

        print(f"  M2 Spearman ρ: {rho_min:.3f} – {rho_max:.3f}  "
              f"(worst at κ={kappa_worst:.1f})")
        print(f"  M2 Accessible GW at baseline (κ=1.0): {gw_base:.1f} GW")
        print(f"  M2 Range across κ scenarios: {gw_worst:.1f} – {gw_best:.1f} GW "
              f"({pct_range:.0f}% span)")

    # M3 summary
    if rtype in m3_oat:
        oat = m3_oat[rtype]
        baseline_gw = oat["baseline_gw"].iloc[0]
        for param in oat["parameter"].unique():
            sub = oat[oat["parameter"] == param]
            lo = sub[sub["scenario_label"] == "Low"]["delta_gw"].values[0]
            hi = sub[sub["scenario_label"] == "High"]["delta_gw"].values[0]
            print(f"  M3 {param:35s}  {lo:+.1f} to {hi:+.1f} GW")
        print(f"  M3 Baseline accessible capacity: {baseline_gw:.1f} GW")

print()
